# Classical ML - Submission Generation on Google Colab

This notebook reproduces the submission files from checkpointed model predictions.

**Required files to upload:**
- `data/train.csv`
- `data/test.csv`
- `data/sample_submission.csv`
- All `.npy` files from `results/` directory (12 files: 6 oof_*.npy + 6 pred_*.npy)

In [ ]:
# Install dependencies
!pip install pandas numpy scikit-learn scipy -q

In [ ]:
# Upload data files
from google.colab import files
print("Upload train.csv, test.csv, sample_submission.csv:")
uploaded = files.upload()

import os
os.makedirs('data', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('submissions', exist_ok=True)

for fn, content in uploaded.items():
    if fn in ['train.csv', 'test.csv', 'sample_submission.csv']:
        with open(f'data/{fn}', 'wb') as f:
            f.write(content)
        print(f'Saved data/{fn}')

In [ ]:
# Upload .npy files from results/
print("Upload all 12 .npy files from results/:")
print("  oof_lgbm_c10_s42.npy, oof_lgbm_c10_s43.npy, oof_lgbm_c20_s42.npy")
print("  oof_xgb_d6_s42.npy, oof_xgb_d6_s43.npy, oof_hgb_s42.npy")
print("  pred_lgbm_c10_s42.npy, pred_lgbm_c10_s43.npy, pred_lgbm_c20_s42.npy")
print("  pred_xgb_d6_s42.npy, pred_xgb_d6_s43.npy, pred_hgb_s42.npy")

uploaded = files.upload()

for fn, content in uploaded.items():
    if fn.endswith('.npy'):
        with open(f'results/{fn}', 'wb') as f:
            f.write(content)
        print(f'Saved results/{fn}')

In [ ]:
# Verify all files present
import os

required_data = ['data/train.csv', 'data/test.csv', 'data/sample_submission.csv']
required_npy = [
    'results/oof_lgbm_c10_s42.npy', 'results/oof_lgbm_c10_s43.npy', 'results/oof_lgbm_c20_s42.npy',
    'results/oof_xgb_d6_s42.npy', 'results/oof_xgb_d6_s43.npy', 'results/oof_hgb_s42.npy',
    'results/pred_lgbm_c10_s42.npy', 'results/pred_lgbm_c10_s43.npy', 'results/pred_lgbm_c20_s42.npy',
    'results/pred_xgb_d6_s42.npy', 'results/pred_xgb_d6_s43.npy', 'results/pred_hgb_s42.npy'
]

print("Data files:")
for f in required_data:
    print(f"  {f}: {'✓' if os.path.exists(f) else '✗ MISSING'}")

print("\nNPY files:")
for f in required_npy:
    print(f"  {f}: {'✓' if os.path.exists(f) else '✗ MISSING'}")

In [ ]:
# blend_utils.py
import numpy as np
from scipy.optimize import nnls
from sklearn.metrics import roc_auc_score

def load_oof_pred(results_dir, tags):
    oof = {t: np.load(f'{results_dir}/oof_{t}.npy') for t in tags}
    pred = {t: np.load(f'{results_dir}/pred_{t}.npy') for t in tags}
    return oof, pred

def select_blend(y, tags, oof, pred, log=print):
    singles = {t: roc_auc_score(y, oof[t]) for t in tags}
    for t in sorted(singles, key=singles.get, reverse=True):
        log(f'  single {t}: {singles[t]:.5f}')

    best_single = max(singles, key=singles.get)
    best_auc, best_name, best_pred = singles[best_single], best_single, pred[best_single]

    # best pair
    for i, a in enumerate(tags):
        for b in tags[i + 1:]:
            for w in np.linspace(0, 1, 101):
                auc = roc_auc_score(y, w * oof[a] + (1 - w) * oof[b])
                if auc > best_auc:
                    best_auc = auc
                    best_name = f'pair {a}+{b} w={w:.2f}'
                    best_pred = w * pred[a] + (1 - w) * pred[b]

    # NNLS over all OOFs
    O = np.column_stack([oof[t] for t in tags])
    w_nnls, _ = nnls(O, y.astype(float))
    w_norm = w_nnls / w_nnls.sum()
    auc_nnls = roc_auc_score(y, O @ w_norm)
    if auc_nnls > best_auc:
        best_auc = auc_nnls
        best_name = 'nnls ' + ', '.join(
            f'{t}:{w:.2f}' for t, w in zip(tags, w_norm) if w > 0.01)
        best_pred = np.column_stack([pred[t] for t in tags]) @ w_norm

    return best_name, best_auc, best_pred, singles, best_single

In [ ]:
# Run submission generation
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

TAGS = ['lgbm_c10_s42', 'lgbm_c10_s43', 'lgbm_c20_s42', 'xgb_d6_s42',
        'xgb_d6_s43', 'hgb_s42']

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
ss = pd.read_csv('data/sample_submission.csv')
y = train['target'].values

oof, pred = load_oof_pred('results', TAGS)
name, auc, best_pred, singles, best_single = select_blend(y, TAGS, oof, pred)
print(f'selected blend: {name}\nOOF AUC = {auc:.5f}')

# lgbm-only reference submission
lgbm_tags = [t for t in TAGS if t.startswith('lgbm')]
lgbm_best = max(lgbm_tags, key=lambda t: singles[t])
lgbm_auc, lgbm_pred = singles[lgbm_best], pred[lgbm_best]
if len(lgbm_tags) >= 2:
    for i, a in enumerate(lgbm_tags):
        for b in lgbm_tags[i + 1:]:
            for w in np.linspace(0, 1, 101):
                auc2 = roc_auc_score(y, w * oof[a] + (1 - w) * oof[b])
                if auc2 > lgbm_auc:
                    lgbm_auc = auc2
                    lgbm_best = f'pair {a}+{b} w={w:.2f}'
                    lgbm_pred = w * pred[a] + (1 - w) * pred[b]
print(f'lgbm submission: {lgbm_best}  OOF AUC = {lgbm_auc:.5f}')

def checks(name, p):
    assert len(p) == len(test), 'row count mismatch'
    assert ((p > 0) & (p < 1)).all(), 'probability out of range'
    assert not np.isnan(p).any(), 'NaN in predictions'
    print(f'checks passed: {name}')

checks('blend', best_pred)
checks('lgbm', lgbm_pred)

pd.DataFrame({'id': test['id'], 'target': best_pred}).to_csv(
    'submissions/submission_blend.csv', index=False)
pd.DataFrame({'id': test['id'], 'target': lgbm_pred}).to_csv(
    'submissions/submission_lgbm.csv', index=False)

for f in ('submissions/submission_blend.csv', 'submissions/submission_lgbm.csv'):
    sub = pd.read_csv(f)
    assert list(sub.columns) == list(ss.columns), f'{f}: column mismatch'
    assert (sub['id'] == ss['id']).all(), f'{f}: id order mismatch'
print('submission format validated against sample_submission')

print('\nDone! Files saved to submissions/')

In [ ]:
# Download the generated submissions
from google.colab import files
files.download('submissions/submission_blend.csv')
files.download('submissions/submission_lgbm.csv')